In [1]:
import numpy as np
from load import *
from aligner import *


[*********************100%***********************]  5 of 5 completed


# 2. Validate

In [2]:
# Missing values
data.isna().sum()

Price   Ticker
Close   GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
High    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Low     GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Open    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Volume  GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
dtype: int64

In [3]:
data.index.duplicated().sum()
print(data.shape)
print(data.head())
print(data.columns)

(6539, 25)
Price      Close                               High                            \
Ticker       GLD        QQQ        SPY TLT VNQ  GLD        QQQ        SPY TLT   
Date                                                                            
2000-01-03   NaN  79.839706  91.132790 NaN NaN  NaN  81.050995  92.895134 NaN   
2000-01-04   NaN  74.362526  87.568886 NaN NaN  NaN  78.786359  90.271146 NaN   
2000-01-05   NaN  72.466614  87.725510 NaN NaN  NaN  75.521166  88.685007 NaN   
2000-01-06   NaN  67.489815  86.315666 NaN NaN  NaN  74.151899  88.665457 NaN   
2000-01-07   NaN  75.837143  91.328575 NaN NaN  NaN  75.837143  91.328575 NaN   

Price           ... Open                               Volume            \
Ticker     VNQ  ...  GLD        QQQ        SPY TLT VNQ    GLD       QQQ   
Date            ...                                                       
2000-01-03 NaN  ...  NaN  81.050995  92.895134 NaN NaN    NaN  36345200   
2000-01-04 NaN  ...  NaN  77.522407  89.

In [4]:
close_prices = data["Close"]

for ticker in close_prices.columns:
    print(
        ticker,
        close_prices[ticker].first_valid_index(),
        close_prices[ticker].last_valid_index()
    )

GLD 2004-11-18 00:00:00 2025-12-31 00:00:00
QQQ 2000-01-03 00:00:00 2025-12-31 00:00:00
SPY 2000-01-03 00:00:00 2025-12-31 00:00:00
TLT 2002-07-30 00:00:00 2025-12-31 00:00:00
VNQ 2004-09-29 00:00:00 2025-12-31 00:00:00


# 3. Align assets

In [5]:
# We need to align so all 4 ETF:s start at the same time to match future matrix
# For the moment, we work with only one column
close_prices = data["Close"]

aligned_prices = align_assets(close_prices)
print(aligned_prices.head())
print(aligned_prices.shape)
aligned_prices.isna().sum().sum()

Ticker            GLD        QQQ        SPY        TLT        VNQ
Date                                                             
2004-11-18  44.380001  33.120064  79.745255  43.988655  21.189310
2004-11-19  44.779999  32.605846  78.858772  43.637638  20.970291
2004-11-22  44.950001  32.917751  79.234863  43.865051  21.115013
2004-11-23  44.750000  32.867172  79.355751  43.919426  21.204958
2004-11-24  45.049999  33.153797  79.543770  43.919426  21.572582
(5313, 5)


np.int64(0)

# 4. Convert raw to log returns

In [6]:
# Using r_t = log(P_t / P_{t-1})
log_returns = np.log(aligned_prices / aligned_prices.shift(1))

In [7]:
# Remove the first NaN raw
log_returns = log_returns.dropna()
print(log_returns.shape)
log_returns.describe()

(5312, 5)


Ticker,GLD,QQQ,SPY,TLT,VNQ
count,5312.000000,5312.000000,5312.000000,5312.000000,5312.000000
mean,0.000412,0.000549,0.000403,0.000125,0.000265
std,0.011132,0.013615,0.011983,0.009230,0.018122
min,-0.091905,-0.127592,-0.115887,-0.069010,-0.217084
25%,-0.005113,-0.005178,-0.003944,-0.005346,-0.006470
50%,0.000588,0.001146,0.000718,0.000445,0.000768
75%,0.006217,0.007321,0.005776,0.005504,0.007510
max,0.106974,0.114799,0.135578,0.072503,0.157060


# 5. Create rolling observation window

In [9]:
# Seeing only today's return is not relevant enough
# We also care to see how turbulent has the asset been recently
# we compute the std_dev or returns σ_t=std(r_t−19,...,r_t), 20 is approx. 1 month of trades
volatility_20 = log_returns.rolling(20).std()
print(volatility_20.head(25)) # volatility = 0.009785 ==> 0.97%

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

# 6. Generate derived features

In [12]:
# 7. Normalize

# 8. Create Splits
